# 🧩 토큰화 비교 실습 · Tokenization Comparison Lab

**한국어:** 이 노트북은 같은 문장을 **영어와 한국어**로 토큰화하여 토큰 수를 비교합니다. 코딩 지식이 없어도 됩니다 — 위 메뉴에서 **런타임 → 모두 실행 (Runtime → Run all)** 을 누르고 결과를 보세요.

**English:** This notebook tokenizes the same sentence in **English and Korean** and compares the token counts. No coding needed — just click **Runtime → Run all** in the menu above and watch the output.

세 가지 **실제** 토크나이저를 사용합니다 / We use three **real** tokenizers:
- **GPT-2** — Byte-Pair Encoding (BPE)
- **BERT (multilingual)** — WordPiece (이어지는 조각을 `##` 로 표시 / marks continuations with `##`)
- **XLM-RoBERTa** — SentencePiece (`▁` = 공백 / space)

## 0) 준비 / Setup
필요한 라이브러리를 설치합니다 (~20초). / Install libraries (~20s).

In [ ]:
!pip install -q transformers sentencepiece
print("✅ 설치 완료 / done")

## 1) 문장 입력 / Enter your sentences
아래 두 줄을 자유롭게 바꿔 보세요 (따옴표 안의 글자만 수정).
Feel free to edit the two lines below (only the text inside the quotes).

In [ ]:
# ✏️ 여기를 수정하세요 / Edit here
english_sentence = "Artificial intelligence is changing the world."
korean_sentence  = "인공지능이 세상을 바꾸고 있습니다."

print("EN:", english_sentence)
print("KR:", korean_sentence)

## 2) 토크나이저 불러오기 / Load the tokenizers
모델 가중치가 아니라 **토크나이저만** 내려받으므로 빠릅니다.
This only downloads the **tokenizers** (not the heavy model weights), so it's fast.

In [ ]:
from transformers import AutoTokenizer

tokenizers = {
    "GPT-2 (BPE)":            AutoTokenizer.from_pretrained("gpt2"),
    "BERT-multi (WordPiece)": AutoTokenizer.from_pretrained("bert-base-multilingual-cased"),
    "XLM-R (SentencePiece)":  AutoTokenizer.from_pretrained("xlm-roberta-base"),
}
print("✅ 불러오기 완료 / loaded:", ", ".join(tokenizers.keys()))

## 3) 토큰 비교 / Compare the tokens
각 토크나이저가 문장을 어떻게 조각내는지, 토큰이 몇 개인지 봅니다.
See how each tokenizer splits the text and how many tokens it produces.

> 💡 `##` (BERT) = 앞 조각에 이어 붙음 / continues previous piece.  `▁` (XLM-R) = 단어 앞 공백 / a space before a word.

In [ ]:
def show(text, title):
    print("=" * 64)
    print(f"{title}: {text}")
    print("=" * 64)
    for name, tok in tokenizers.items():
        pieces = tok.tokenize(text)
        print(f"\n[{name}]  →  {len(pieces)} tokens / 토큰")
        print("   ", pieces)
    print()

show(english_sentence, "ENGLISH / 영어")
show(korean_sentence,  "KOREAN / 한국어")

## 4) 토큰세 / The "token tax"
같은 뜻인데 한국어가 토큰을 더 많이 쓰나요? 비율(KR/EN)을 계산합니다.
Same meaning — does Korean use more tokens? Let's compute the ratio (KR/EN).

In [ ]:
print(f"{'Tokenizer':28}{'EN':>5}{'KR':>5}{'KR/EN':>8}")
print("-" * 46)
for name, tok in tokenizers.items():
    en = len(tok.tokenize(english_sentence))
    kr = len(tok.tokenize(korean_sentence))
    ratio = kr / en if en else 0
    print(f"{name:28}{en:>5}{kr:>5}{ratio:>7.1f}x")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

names     = list(tokenizers.keys())
en_counts = [len(tok.tokenize(english_sentence)) for tok in tokenizers.values()]
kr_counts = [len(tok.tokenize(korean_sentence))  for tok in tokenizers.values()]

x = np.arange(len(names)); w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, en_counts, w, label="English")
ax.bar(x + w/2, kr_counts, w, label="Korean")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=12, ha="right")
ax.set_ylabel("number of tokens")
ax.set_title("Token count: English vs Korean (same meaning)")
ax.legend()
for i, (e, k) in enumerate(zip(en_counts, kr_counts)):
    ax.text(i - w/2, e + 0.1, str(e), ha="center")
    ax.text(i + w/2, k + 0.1, str(k), ha="center")
plt.tight_layout(); plt.show()

## 5) 저자원 언어 (보너스) / Low-resource languages (bonus)
같은 문장을 여러 언어로 측정합니다 — 학습 데이터가 적은 언어일수록 토큰이 많아집니다.
The same sentence across many languages — fewer training data usually means more tokens.

In [ ]:
sentences = {
    "English": "Artificial intelligence is changing the world.",
    "Spanish": "La inteligencia artificial está cambiando el mundo.",
    "Chinese": "人工智能正在改变世界。",
    "Korean":  "인공지능이 세상을 바꾸고 있습니다.",
    "Hindi":   "कृत्रिम बुद्धिमत्ता दुनिया को बदल रही है।",
    "Swahili": "Akili bandia inabadilisha ulimwengu.",
    "Amharic": "ሰው ሰራሽ አስተዋይነት ዓለምን እየቀየረ ነው።",
}
tok = tokenizers["GPT-2 (BPE)"]
counts = {lang: len(tok.tokenize(t)) for lang, t in sentences.items()}
base = counts["English"]
print(f"{'Language':10}{'tokens':>8}{'vs English':>12}")
print("-" * 30)
for lang, c in sorted(counts.items(), key=lambda kv: kv[1]):
    print(f"{lang:10}{c:>8}{c/base:>11.1f}x")

## 무엇을 관찰했나요? / What did you notice?

- 한국어는 같은 뜻이라도 **토큰이 더 많습니다** → 더 비싸고 느립니다.
  Korean uses **more tokens** for the same meaning → it costs more and is slower.
- **WordPiece** (BERT) 는 이어지는 조각을 `##` 로 표시합니다 / marks continuations with `##`.
- **SentencePiece** (XLM-R) 는 공백을 `▁` 로 표시합니다 / marks spaces with `▁`.
- **GPT-2 (BPE)** 는 한국어를 바이트로 쪼개 이상한 기호가 보일 수 있습니다 / may show odd byte symbols for Korean.
- 학습 데이터가 적은 **저자원 언어**일수록 토큰세(token tax)가 큽니다 / lower-resource languages pay a bigger token tax.

---
*동반 자료 / Companion materials: the interactive web app + the printable glossary & worksheet.*
*Grounded in “Natural Language Processing with Transformers” (Tunstall, von Werra & Wolf, O'Reilly 2022).*